# RSEc MCP server: replaying a demo session

In [`docs/demo.md`](../docs/demo.md) a local LLM (Qwen, in LM Studio) uses the MCP server to answer three questions about metagenomics tools.
An LLM session cannot be repeated exactly, so this notebook **replays the same scenario with a fixed script**.
The questions and the shortlist of tools are written down; every answer below comes from a real call to the MCP tools of `server.py`, made in-process as an MCP client would.

1. *"I need to perform i) quality control on metagenomics data, and ii) quantify the abundance of species. Which tools can I use? Provide the bio.tools URLs and associated EDAM annotations."*
2. *"For all these tools, display their license."*
3. *"Also list their inputs and outputs, in terms of EDAM annotations."*

The shortlist of the session is FastQC, MultiQC, Kraken2, Bracken, MetaPhlAn and QIIME 2. The notebook checks it against the graph.
The first tool call loads about 740,000 triples (about 10 s once the files are downloaded); later calls are SPARQL queries on the loaded graph.
The whole notebook runs in about a minute.

In [18]:
import os, sys, time

os.environ.setdefault("RSEC_LOG_LEVEL", "WARNING")  # keep the loading logs out of the notebook
sys.path.insert(0, "..")                            # server.py lives in mcp/

import pandas as pd
from fastmcp import Client

import server

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)

CALLS = []  # one entry per tool call, summarised at the end


async def tool(tool_name, /, **arguments):
    """Call one MCP tool and return its result, as an agent would."""
    t0 = time.perf_counter()
    async with Client(server.mcp) as client:
        result = (await client.call_tool(tool_name, arguments)).data
    CALLS.append({"tool": tool_name, "seconds": time.perf_counter() - t0})
    return result


# The shortlist of the LM Studio session (bio.tools ids are case-sensitive).
CANDIDATES = ["fastqc", "multiqc", "kraken2", "bracken", "metaphlan", "qiime2"]

## The tools the agent sees

The server describes its tools to the client (name, arguments, documentation). This is all the agent knows about the knowledge graph:
there is no SPARQL to write, and the agent cannot send a query of its own.

In [19]:
async with Client(server.mcp) as client:
    catalogue = await client.list_tools()

pd.DataFrame(
    [{"tool": t.name, "arguments": ", ".join(t.input_schema["properties"])} for t in catalogue]
)

,tool,arguments
0,resolve_edam_term,"text, kind, limit"
1,get_direct_term_parents,term
2,get_direct_term_child,term
3,search_software_by_topic,"topic, limit, offset"
4,search_software_by_operation,"operation, limit, offset"
5,get_license,tool
6,get_software_io,tool
7,describe_software,name


## 1. Which tools, and what are their EDAM annotations?

### Step 1: from words to EDAM terms

Software is searched by EDAM id, so the agent first turns the needs into terms with `resolve_edam_term`.

The strategy is a lexical match on the EDAM ontology, done by one SPARQL query:

- A term is a candidate when its label, one of its synonyms or its definition contains **every word** of the request, as whole words and in any order. Deprecated terms are left out.
- Candidates are ranked by where the words are found: exact label, exact synonym, words in the label, words in a synonym, words in the definition (the `matched` column). Ties go to the shorter label.
- `kind` restricts the answer to one branch of EDAM (`topic`, `operation`, `data` or `format`). Without it, the same label can come back for several branches, as with *Taxonomic classification* below.

There is no semantic or fuzzy matching: "abundance estimation" finds nothing, because no EDAM term contains both words, so the agent has to rephrase (here, "species frequency").

In [20]:
needs = ["sequencing quality control", "taxonomic classification", "species frequency", "metagenomics"]

rows = []
for text in needs:
    found = await tool("resolve_edam_term", text=text, limit=3)
    for r in found["results"]:
        rows.append({"asked": text, "id": r["id"], "label": r["label"], "kind": r["kind"], "matched": r["matched"]})

pd.DataFrame(rows)

,asked,id,label,kind,matched
0,sequencing quality control,operation_3218,Sequencing quality control,operation,label
1,taxonomic classification,data_1872,Taxonomic classification,data,label
2,taxonomic classification,operation_3460,Taxonomic classification,operation,label
3,species frequency,operation_3221,Species frequency estimation,operation,label
4,metagenomics,topic_3174,Metagenomics,topic,label
5,metagenomics,operation_3731,Sample comparison,operation,definition
6,metagenomics,operation_3744,Multiple sample visualisation,operation,definition


### Step 2: software annotated with these terms

`search_software_by_operation` and `search_software_by_topic` list the tools annotated with a term or one of its narrower terms, 100 at a time.
The agent reads every page, then checks whether the shortlisted tools are in the lists.

In [21]:
async def all_software(kind, term_id):
    """All the tools annotated with an EDAM operation or topic, page after page."""
    found, offset = {}, 0
    while True:
        page = await tool(f"search_software_by_{kind}", **{kind: term_id}, limit=100, offset=offset)
        found.update({r["biotools_id"]: r["name"] for r in page["results"]})
        if page["count"] < page["limit"]:
            return found
        offset += page["limit"]


SEARCHES = {
    "Sequencing quality control": ("operation", "operation_3218"),
    "Taxonomic classification": ("operation", "operation_3460"),
    "Species frequency estimation": ("operation", "operation_3221"),
    "Metagenomics (topic)": ("topic", "topic_3174"),
}
listed = {name: await all_software(kind, term_id) for name, (kind, term_id) in SEARCHES.items()}

print("Tools annotated with each term (narrower terms included):")
print(pd.Series({name: len(ids) for name, ids in listed.items()}).to_string())

pd.DataFrame(
    {name: {t: "yes" if t in ids else "" for t in CANDIDATES} for name, ids in listed.items()}
).rename_axis("shortlisted tool")

Tools annotated with each term (narrower terms included):
Sequencing quality control       518
Taxonomic classification         318
Species frequency estimation      15
Metagenomics (topic)            1054


,Sequencing quality control,Taxonomic classification,Species frequency estimation,Metagenomics (topic)
shortlisted tool,,,,
fastqc,yes,,,
multiqc,yes,,,
kraken2,,yes,,yes
bracken,,,,yes
metaphlan,,yes,,yes
qiime2,,yes,,yes


### Step 3: the annotations of each shortlisted tool

`describe_software` returns everything the graph holds about a tool (a SPARQL `DESCRIBE`): EDAM topics and operations come as IRIs.
The agent gets their labels with `get_direct_term_parents`, which gives the label of the term it is asked about.

In [23]:
LABELS = {}


async def label(term_id):
    if term_id not in LABELS:
        LABELS[term_id] = (await tool("get_direct_term_parents", term=term_id))["term"]["label"]
    return LABELS[term_id]


async def labelled(iris):
    ids = sorted(iri.rsplit("/", 1)[-1] for iri in iris)
    return "; ".join([f"{i} ({await label(i)})" for i in ids])


rows = []
for t in CANDIDATES:
    described = await tool("describe_software", name=t)
    assert described["count"] == 1, t
    record = described["tools"][0]
    props = record["properties"]
    rows.append({
        "tool": props["schema:name"][0],
        "bio.tools URL": record["iri"],
        "operations": await labelled(props.get("schema:featureList", [])),
        "topics": await labelled(props.get("schema:applicationSubCategory", [])),
    })

pd.DataFrame(rows)

,tool,bio.tools URL,operations,topics
0,FastQC,https://bio.tools/fastqc,operation_0236 (Sequence composition calculation); operation_2238 (Statistical calculation); operation_3218 (Sequencing quality control),topic_0080 (Sequence analysis); topic_3168 (Sequencing); topic_3572 (Data quality management)
1,MultiQC,https://bio.tools/multiqc,operation_2428 (Validation); operation_3218 (Sequencing quality control),topic_0080 (Sequence analysis); topic_0091 (Bioinformatics); topic_0622 (Genomics); topic_3168 (Sequencing)
2,kraken2,https://bio.tools/kraken2,operation_3460 (Taxonomic classification),topic_0637 (Taxonomy); topic_3174 (Metagenomics); topic_3301 (Microbiology); topic_3697 (Microbial ecology)
3,Bracken,https://bio.tools/bracken,operation_2238 (Statistical calculation),topic_3174 (Metagenomics); topic_3697 (Microbial ecology)
4,MetaPhlAn,https://bio.tools/metaphlan,operation_3460 (Taxonomic classification),topic_0194 (Phylogenomics); topic_3174 (Metagenomics)
5,QIIME 2,https://bio.tools/qiime2,operation_0337 (Visualisation); operation_3460 (Taxonomic classification); operation_3933 (Demultiplexing),topic_3174 (Metagenomics); topic_3697 (Microbial ecology); topic_3941 (Metatranscriptomics)


**What to notice.** The graph gives the exact labels of the EDAM ids, so the annotations can be checked:

- FastQC and MultiQC are annotated with *Sequencing quality control*, but not with the topic *Metagenomics*: they are general-purpose tools. An agent that intersects "quality control" and "metagenomics" would not find them.
- Bracken has neither *Taxonomic classification* nor *Species frequency estimation*; it is found through its topic (*Metagenomics*) only.
- Some labels in the LM Studio transcript are not the labels of the graph. For example `operation_2238` is *Statistical calculation* (not "Abundance estimation"), `topic_3168` is *Sequencing* (not "Metagenomics"), `operation_3933` is *Demultiplexing* (not "Relative abundance"). Calling the tools gives the labels as they are in EDAM.

## 2. Licenses

`get_license` gives the license text as written in the data (free text, possibly empty).

In [24]:
rows = []
for t in CANDIDATES:
    result = await tool("get_license", tool=t)
    rows.append({"tool": result["name"], "license": ", ".join(result["licenses"]) or "(none)"})

pd.DataFrame(rows)

,tool,license
0,FastQC,GPL-3.0
1,MultiQC,GPL-3.0
2,kraken2,MIT
3,Bracken,GPL-3.0
4,MetaPhlAn,MIT
5,QIIME 2,BSD-3-Clause


## 3. Inputs and outputs, in EDAM terms

`get_software_io` gives each input and output with its EDAM data type and formats.
When a tool has no annotation, the answer says so instead of guessing.

In [25]:
rows = []
for t in CANDIDATES:
    io = await tool("get_software_io", tool=t)
    for direction in ("inputs", "outputs"):
        for p in io[direction]:
            rows.append({
                "tool": io["name"],
                "direction": direction[:-1],
                "parameter": p["name"],
                "EDAM data": f'{p["data"]["id"]} ({p["data"]["label"]})',
                "EDAM formats": "; ".join(f'{f["id"]} ({f["label"]})' for f in p["formats"]),
            })
        if not io[direction]:
            rows.append({"tool": io["name"], "direction": direction[:-1], "parameter": "(not annotated)",
                         "EDAM data": "", "EDAM formats": ""})

pd.DataFrame(rows)

,tool,direction,parameter,EDAM data,EDAM formats
0,FastQC,input,Raw sequence,data_0848 (Raw sequence),format_1930 (FASTQ); format_2182 (FASTQ-like format (text)); format_2572 (BAM); format_2573 (SAM)
1,FastQC,output,Sequence report,data_2955 (Sequence report),format_2331 (HTML)
2,MultiQC,input,(not annotated),,
3,MultiQC,output,Quality control report,data_3914 (Quality control report),format_2331 (HTML); format_3464 (JSON); format_3475 (TSV); format_3750 (YAML); format_3752 (CSV)
4,MultiQC,output,Sequence alignment report,data_0867 (Sequence alignment report),format_2331 (HTML)
5,kraken2,input,DNA sequence,data_3494 (DNA sequence),format_1929 (FASTA); format_1930 (FASTQ)
6,kraken2,output,Taxonomy,data_3028 (Taxonomy),format_3475 (TSV)
7,Bracken,input,(not annotated),,
8,Bracken,output,(not annotated),,
9,MetaPhlAn,input,DNA sequence,data_3494 (DNA sequence),format_1929 (FASTA); format_1930 (FASTQ); format_2573 (SAM)


Bracken and QIIME 2 have no input/output annotation in the RSEc data, and MultiQC has outputs only: an agent has to say so, which is what the `note` of `get_software_io` is for. MetaPhlAn's output is listed twice because its record holds two output parameters with the same type and format.

## The tool calls of this notebook

Everything above came from these calls. The first one waits for the data to load; the others are SPARQL queries on the loaded graph.

In [26]:
(
    pd.DataFrame(CALLS)
    .groupby("tool")
    .agg(calls=("seconds", "count"), total_seconds=("seconds", "sum"), slowest_seconds=("seconds", "max"))
    .round(2)
    .sort_values("total_seconds", ascending=False)
)

,calls,total_seconds,slowest_seconds
tool,,,
describe_software,12,17.92,1.54
search_software_by_operation,11,7.84,1.42
search_software_by_topic,11,4.56,0.77
resolve_edam_term,4,4.03,1.02
get_license,6,0.59,0.54
get_direct_term_parents,36,0.24,0.01
get_software_io,6,0.06,0.01


## Run it again

```bash
cd mcp
./run_tests.sh --notebook      # rebuild and execute this notebook
uv run --with jupyterlab --with pandas jupyter lab notebooks/demo.ipynb   # explore it
```

The notebook is generated: edit `notebooks/build_demo.py` rather than the `.ipynb`. To plug the server into an LLM client such as LM Studio, see the README.